In [9]:
import os 


In [10]:
%pwd

'd:\\NLP Project\\research'

In [11]:
os.chdir("../")

In [12]:
%pwd

'd:\\NLP Project'

In [2]:
from NexText.constants import *
from NexText.utils.common import read_yaml, create_directories
from NexText.entity.config_entity import ModelTrainerConfig

In [3]:
class ConfigurationManager:

    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:

        config = self.config.model_trainer
        params = self.params.TrainingArguments

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_ckpt=config.model_ckpt,
            num_train_epochs=int(params.num_train_epochs),
            warmup_steps=int(params.warmup_steps),
            per_device_train_batch_size=int(params.per_device_train_batch_size),
            per_device_eval_batch_size=int(params.get("per_device_eval_batch_size", params.per_device_train_batch_size)),
            weight_decay=float(params.weight_decay),
            logging_steps=int(params.logging_steps),
            evaluation_strategy=str(params.evaluation_strategy),
            eval_steps=int(params.eval_steps),
            save_steps=int(float(params.save_steps)),
            gradient_accumulation_steps=int(params.gradient_accumulation_steps)
        )

        return model_trainer_config

In [4]:
import os
import torch

from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
)

from datasets import load_from_disk

from NexText.logging import logger
from NexText.entity.config_entity import ModelTrainerConfig
from NexText.config.configuration import ConfigurationManager

d:\NLP Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
class ModelTrainer:

    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):

        # Load tokenizer
        tokenizer = AutoTokenizer.from_pretrained(
            self.config.model_ckpt
        )

        # Load Pegasus model
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(
            self.config.model_ckpt
        )

        # Data collator
        seq2seq_data_collator = DataCollatorForSeq2Seq(
            tokenizer=tokenizer,
            model=model_pegasus
        )

        # Load transformed dataset
        logger.info("Loading transformed dataset...")

        dataset_samsum_pt = load_from_disk(
            self.config.data_path
        )

        logger.info(
            f"Train samples: {len(dataset_samsum_pt['train'])}"
        )

        logger.info(
            f"Validation samples: "
            f"{len(dataset_samsum_pt['validation'])}"
        )

        # Training arguments
        trainer_args = TrainingArguments(
            output_dir=self.config.root_dir,
            num_train_epochs=self.config.num_train_epochs,
            warmup_steps=self.config.warmup_steps,
            per_device_train_batch_size=(
                self.config.per_device_train_batch_size
            ),
            per_device_eval_batch_size=self.config.per_device_eval_batch_size,
                self.config.per_device_train_batch_size
            ),
            weight_decay=self.config.weight_decay,
            logging_steps=self.config.logging_steps,
            eval_strategy=self.config.evaluation_strategy,
            eval_steps=self.config.eval_steps,
            save_steps=self.config.save_steps,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps,
            gradient_checkpointing=True,
            optim="adafactor",
            ),
            report_to="none",
            # Use FP16 if CUDA is available
            fp16=torch.cuda.is_available(),
        )

        # Trainer
        trainer = Trainer(
            model=model_pegasus,
            args=trainer_args,
            processing_class=tokenizer,
            data_collator=seq2seq_data_collator,
            train_dataset=dataset_samsum_pt["train"],
            eval_dataset=dataset_samsum_pt["validation"],
        )

        logger.info("Starting model training...")

        trainer.train()

        logger.info("Model training completed.")

        # Save model
        model_path = os.path.join(
            self.config.root_dir,
            "pegasus-samsum-model"
        )

        model_pegasus.save_pretrained(model_path)

        # Save tokenizer
        tokenizer_path = os.path.join(
            self.config.root_dir,
            "tokenizer"
        )

        tokenizer.save_pretrained(tokenizer_path)

        logger.info(
            f"Model saved to: {model_path}"
        )

        logger.info(
            f"Tokenizer saved to: {tokenizer_path}"
        )


SyntaxError: unmatched ')' (1850925620.py, line 59)

In [6]:
try:
    config = ConfigurationManager()

    model_trainer_config = (
        config.get_model_trainer_config()
    )

    model_trainer = ModelTrainer(
        config=model_trainer_config
    )

    model_trainer.train()

except Exception as e:
    logger.exception(e)
    raise e

[ 2026-09-23 20:42:00,134 ] 21 NexText_logger - INFO - YAML file: D:\NLP Project\config\config.yaml loaded successfully.
[ 2026-09-23 20:42:00,145 ] 21 NexText_logger - INFO - YAML file: D:\NLP Project\params.yaml loaded successfully.
[ 2026-09-23 20:42:00,148 ] 49 NexText_logger - INFO - Created directory at: artifacts
[ 2026-09-23 20:42:00,149 ] 49 NexText_logger - INFO - Created directory at: artifacts/model_trainer
[ 2026-09-23 20:42:00,150 ] 15 NexText_logger - ERROR - name 'ModelTrainer' is not defined
Traceback (most recent call last):
  File "C:\Users\HP\AppData\Local\Temp\ipykernel_19248\1289382499.py", line 8, in <module>
    model_trainer = ModelTrainer(
                    ^^^^^^^^^^^^
NameError: name 'ModelTrainer' is not defined


NameError: name 'ModelTrainer' is not defined